<a href="https://colab.research.google.com/github/genji970/LLM_pretraining/blob/main/pretraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers

# **data preparation**

In [102]:
from dataclasses import dataclass
import string , random

@dataclass
class Data_Config:
  data_path : str = None
  # task of data / purpose of data
  data_kind : str = None
  # Number of data
  sample_num : int = None
  # How many data will be loaded in Dataset
  chunk : int = None
  name : str = None

  token_key : str = None
  token_id : int = None

  data_structure : tuple | None = ("question" , "reference" , "answer")

  # what is field()?

# **processing tokenizer**

In [103]:
import re
import time
from typing import List , Dict

class Tokenizer:
  def __init__(self , data_name : str = None , dictionary : dict[str , int] = {}):
    self.data_name = data_name
    self.dictionary = dictionary
    self.max_count = int(len(self.dictionary)) if self.dictionary is not None else 0

  # tokenize -> encoder
  def tokenize(self,num_samples : list[str | None] = None) -> list[str | None]:

    # need parallel since it is one by one and take long time and it's inefficient
    tokens_list = []
    for text in num_samples:

      # #, ##, ### 제거
      text = re.sub(r"#+", " ", text)

      # 영어 단어와 문장부호를 각각 추출
      raw_tokens = re.findall(
          r"[A-Za-z]+|[.,!?;:()]",
          text,
      )

      tokens = []

      for token in raw_tokens:
        match = re.fullmatch(
          r"([A-Za-z]+?)(ing|ed)",
          token,
          flags=re.IGNORECASE,
        )

        if match:
          tokens.append(match.group(1))
          tokens.append(match.group(2))
        else:
          tokens.append(token)
      tokens_list.extend(tokens)
    return tokens_list

  def tokenizer_encoding(self , text_list : list[str | None]) -> Dict:

    for sample in text_list:
      # since dict is hash table, its O(1) in searching
      if sample not in self.dictionary:
        self.max_count += 1
        self.dictionary[sample] = int(self.max_count)
    return self.dictionary

  def inserting_special_token(self, special_token : str | None = None):
    assert type(special_token) == str, "Warning! special_token you added is not str."
    if isinstance(special_token , str):
      if special_token not in self.dictionary.keys():
        self.dictionary[special_token] = self.max_count + 1
        self.max_count += 1

  def print_max_num_encoder_dictionary(self):
    print(f"whole number of tokens/index of dictionary : {self.max_count}")

Might be good to mapping natural language to index with semantic criteria

In [104]:
dictionary = {'son' : 1, 'daughter' : 2}
print(len(dictionary))
tokenizer = Tokenizer(dictionary=dictionary)
tokens=tokenizer.tokenize(['I want to go to school but yesterday there was ##.','I want to go to school but yesterday there was ##.'])
dic_token=tokenizer.tokenizer_encoding(tokens)
print(dic_token)
tokenizer.print_max_num_encoder_dictionary()
type(dic_token['there'])

2
{'son': 1, 'daughter': 2, 'I': 3, 'want': 4, 'to': 5, 'go': 6, 'school': 7, 'but': 8, 'yesterday': 9, 'there': 10, 'was': 11, '.': 12}
whole number of tokens/index of dictionary : 12


int

# **processing positional embedding**

q, k: [batch_size, num_heads, sequence_length, head_dim]

In [125]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision , torchaudio

class RoPE_Positional_Embedding(nn.Module):
  def __init__(self , batch_num, embed_dim, context_length , head_dim):
    super().__init__()
    self.batch_num = batch_num
    self.embed_dim = embed_dim
    self.context_length = context_length
    self.head_dim = head_dim

    if head_dim % 2 != 0:
      raise ValueError("head_dim should be even")

    if embed_dim % head_dim != 0:
      raise ValueError("embed_dim should be divided by head_dim")

    p = torch.arange(0, context_length , 1, dtype=torch.float32)
    p=p[None,None,:,None]

    i=torch.arange(0, head_dim , 2, dtype = torch.float32)

    tmp_frequency = 10000 ** (((-1) * i) / head_dim)

    tmp_frequency=tmp_frequency[None,None,None,:]

    # frequency *=p , frequency might not change its shape so it could make error. need to use new variable.
    frequency = (tmp_frequency * p).repeat(int(batch_num), int(embed_dim//head_dim),int(1),int(1))

    self.register_buffer(
      "frequency",
      frequency,
      persistent=False,
    )

  def transformation(self):
    return torch.sin(self.frequency) , torch.cos(self.frequency)

  def construct_embedding(self,x, sin, cos):
    x_even=x[...,0::2]
    x_odd=x[...,1::2]

    rotate_even=x_even * cos - x_odd * sin
    rotate_odd=x_even * cos + x_odd * sin

    return torch.stack([rotate_even,rotate_odd],dim=-1).flatten(-2)

In [126]:
import torch
batch_num=4
embed_dim=32
head_dim = 16
context_length = 4

"""
p = torch.arange(0, context_length , 1, dtype=torch.float32)
p=p[None,None,:,None]

i=torch.arange(0, head_dim , 2, dtype = torch.float32)

frequency = 10000 ** (((-1) * i) / head_dim)
frequency=frequency[None,None,None,:]
new=p*frequency
"""

positional_embedding=RoPE_Positional_Embedding(batch_num,embed_dim,context_length,head_dim)
positional_embedding_0, positional_embedding_1 = positional_embedding.transformation()
print(positional_embedding_0.shape , positional_embedding_1.shape)

torch.Size([4, 2, 4, 8]) torch.Size([4, 2, 4, 8])


# **Constructing Attention**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision, torchaudio

# **self.attention block**

In [132]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision, torchaudio

# not yet for multi-head attention,
class Self_Attention(nn.Module):

  def __init__(self, batch_num , embed_dim, context_length, num_head):
    super().__init__()
    self.batch_num = batch_num
    self.embed_dim = embed_dim
    self.context_length=context_length
    self.num_head = num_head
    self.head_dim = embed_dim // num_head

    if embed_dim % num_head != 0:
      raise ValueError("embed_dim % num_head should be zero")

    self.positional_embedding = RoPE_Positional_Embedding(batch_num, embed_dim, context_length , embed_dim//num_head)

    self.query_embedding =nn.Parameter(torch.randn(self.head_dim, self.head_dim)) # Normal distribution
    self.key_embedding = nn.Parameter(torch.randn(self.head_dim, self.head_dim))
    self.value_embedding = nn.Parameter(torch.randn(self.head_dim, self.head_dim))

  def construct_q_k_v(self, input):
    return input@self.query_embedding ,input@self.key_embedding,input@self.value_embedding

  def normal_attention(self, query, key, value):

    pre_attention_weight = (query @ key.transpose(-1,-2)) / torch.sqrt(torch.tensor(self.head_dim))
    attention_score=(torch.softmax(pre_attention_weight , dim=-1)) @ value
    return attention_score

  def rope_attention(self, query, key, value):
    sin_transformation,cos_transformation=self.positional_embedding.transformation()
    query=self.positional_embedding.construct_embedding(query,sin_transformation,cos_transformation)
    key=self.positional_embedding.construct_embedding(key,sin_transformation,cos_transformation)

    pre_attention_weight = (query @ key.transpose(-1,-2)) / (self.head_dim**0.5)
    attention_score=(torch.softmax(pre_attention_weight , dim=-1)) @ value
    return attention_score

  # construct_q_k_v -> attention -> forward
  def forward(self , x):

    if x.shape != torch.Size([self.batch_num, self.context_length, self.embed_dim]):
      raise ValueError("input data's shape is not what this model wants")

    x=x.view(self.batch_num, self.context_length , self.num_head, self.embed_dim//self.num_head).transpose(1,2).contiguous()

    if x.shape != torch.Size([self.batch_num, self.num_head, self.context_length, self.head_dim]):
      raise ValueError("x.shape must be equal to [batch_num,num_head, context_length , head_dim]")

    query,key,value=self.construct_q_k_v(x)
    attention_score=self.rope_attention(query,key,value)



In [131]:
torch.tensor([[1,2],[3,4]]).shape == torch.tensor([2,2])

torch.Size([2, 2])